# End-to-End Project: Customer Churn Prediction

A classification project with imbalanced data handling and business insights.

## Project Overview

**Objective**: Predict which customers are likely to churn and identify key factors.

**Skills Applied**:
- Imbalanced classification handling (SMOTE, class weights)
- Feature importance and business interpretation
- Threshold optimization for business metrics
- Cost-sensitive learning
- Model explainability

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Sklearn imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, f1_score, accuracy_score,
    precision_score, recall_score
)

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')
np.random.seed(42)

print("Libraries loaded successfully!")

## 1. Data Generation and Loading

We'll create a realistic telecom churn dataset with class imbalance.

In [ ]:
def generate_churn_data(n_samples=5000, churn_rate=0.15, random_state=42):
    """
    Generate synthetic customer churn data with realistic patterns.
    """
    np.random.seed(random_state)
    
    # Customer demographics
    age = np.random.normal(45, 15, n_samples).clip(18, 80)
    gender = np.random.choice(['Male', 'Female'], n_samples)
    
    # Account information
    tenure_months = np.random.exponential(24, n_samples).clip(1, 72)
    contract_type = np.random.choice(
        ['Month-to-Month', 'One Year', 'Two Year'], 
        n_samples, 
        p=[0.55, 0.25, 0.20]
    )
    
    # Service usage
    monthly_charges = np.random.normal(65, 25, n_samples).clip(20, 150)
    total_charges = monthly_charges * tenure_months
    
    # Support interactions
    support_tickets = np.random.poisson(2, n_samples)
    
    # Services
    has_phone = np.random.choice([0, 1], n_samples, p=[0.1, 0.9])
    has_internet = np.random.choice([0, 1], n_samples, p=[0.2, 0.8])
    has_streaming = np.random.choice([0, 1], n_samples, p=[0.5, 0.5])
    has_security = np.random.choice([0, 1], n_samples, p=[0.6, 0.4])
    
    # Payment method
    payment_method = np.random.choice(
        ['Credit Card', 'Bank Transfer', 'Electronic Check', 'Mailed Check'],
        n_samples,
        p=[0.35, 0.25, 0.25, 0.15]
    )
    
    # Calculate churn probability based on features
    churn_prob = np.zeros(n_samples)
    
    # High churn factors
    churn_prob += (contract_type == 'Month-to-Month') * 0.25
    churn_prob += (tenure_months < 12) * 0.15
    churn_prob += (support_tickets > 3) * 0.2
    churn_prob += (payment_method == 'Electronic Check') * 0.1
    churn_prob += (monthly_charges > 80) * 0.1
    
    # Low churn factors
    churn_prob -= (contract_type == 'Two Year') * 0.3
    churn_prob -= (tenure_months > 36) * 0.2
    churn_prob -= (has_security == 1) * 0.1
    
    # Normalize and add noise
    churn_prob = np.clip(churn_prob + np.random.normal(0, 0.1, n_samples), 0, 1)
    
    # Generate churn labels
    threshold = np.percentile(churn_prob, 100 - churn_rate * 100)
    churned = (churn_prob >= threshold).astype(int)
    
    # Create DataFrame
    df = pd.DataFrame({
        'customer_id': range(1, n_samples + 1),
        'age': age.astype(int),
        'gender': gender,
        'tenure_months': tenure_months.astype(int),
        'contract_type': contract_type,
        'monthly_charges': monthly_charges.round(2),
        'total_charges': total_charges.round(2),
        'support_tickets': support_tickets,
        'has_phone': has_phone,
        'has_internet': has_internet,
        'has_streaming': has_streaming,
        'has_security': has_security,
        'payment_method': payment_method,
        'churned': churned
    })
    
    return df


# Generate data
df = generate_churn_data(5000, churn_rate=0.18)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Check class imbalance
print("=== Target Distribution ===")
churn_counts = df['churned'].value_counts()
print(f"Not Churned: {churn_counts[0]} ({churn_counts[0]/len(df)*100:.1f}%)")
print(f"Churned:     {churn_counts[1]} ({churn_counts[1]/len(df)*100:.1f}%)")
print(f"\nImbalance Ratio: {churn_counts[0]/churn_counts[1]:.1f}:1")

## 2. Exploratory Data Analysis

In [ ]:
# Churn distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie(churn_counts.values, labels=['Not Churned', 'Churned'], 
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Churn Distribution')

# Bar chart
axes[1].bar(['Not Churned', 'Churned'], churn_counts.values, 
            color=['#2ecc71', '#e74c3c'])
axes[1].set_ylabel('Count')
axes[1].set_title('Churn Counts')
for i, v in enumerate(churn_counts.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze churn by categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Contract type
contract_churn = df.groupby('contract_type')['churned'].agg(['mean', 'count'])
bars = axes[0, 0].bar(contract_churn.index, contract_churn['mean'] * 100)
axes[0, 0].set_ylabel('Churn Rate (%)')
axes[0, 0].set_title('Churn Rate by Contract Type')
axes[0, 0].tick_params(axis='x', rotation=15)

# Payment method
payment_churn = df.groupby('payment_method')['churned'].mean() * 100
bars = axes[0, 1].bar(payment_churn.index, payment_churn.values)
axes[0, 1].set_ylabel('Churn Rate (%)')
axes[0, 1].set_title('Churn Rate by Payment Method')
axes[0, 1].tick_params(axis='x', rotation=20)

# Tenure distribution
axes[1, 0].hist([df[df['churned']==0]['tenure_months'], df[df['churned']==1]['tenure_months']],
                bins=20, label=['Not Churned', 'Churned'], alpha=0.7)
axes[1, 0].set_xlabel('Tenure (months)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Tenure Distribution by Churn')
axes[1, 0].legend()

# Monthly charges
axes[1, 1].hist([df[df['churned']==0]['monthly_charges'], df[df['churned']==1]['monthly_charges']],
                bins=20, label=['Not Churned', 'Churned'], alpha=0.7)
axes[1, 1].set_xlabel('Monthly Charges ($)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Monthly Charges Distribution by Churn')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by services
services = ['has_phone', 'has_internet', 'has_streaming', 'has_security']

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(services))
width = 0.35

# Churn rate with service vs without
with_service = [df[df[s]==1]['churned'].mean() * 100 for s in services]
without_service = [df[df[s]==0]['churned'].mean() * 100 for s in services]

bars1 = ax.bar(x - width/2, with_service, width, label='Has Service', color='#3498db')
bars2 = ax.bar(x + width/2, without_service, width, label='No Service', color='#e74c3c')

ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Service Subscription')
ax.set_xticks(x)
ax.set_xticklabels([s.replace('has_', '').title() for s in services])
ax.legend()

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Define features
feature_cols = [
    'age', 'tenure_months', 'monthly_charges', 'total_charges', 'support_tickets',
    'has_phone', 'has_internet', 'has_streaming', 'has_security',
    'gender', 'contract_type', 'payment_method'
]

X = df[feature_cols]
y = df['churned']

# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate: {y_test.mean()*100:.1f}%")

In [ ]:
# Create preprocessing pipeline
numerical_features = ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'support_tickets']
binary_features = ['has_phone', 'has_internet', 'has_streaming', 'has_security']
categorical_features = ['gender', 'contract_type', 'payment_method']

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),
    ('categorical', categorical_pipeline, categorical_features),
    ('binary', 'passthrough', binary_features)
])

print("Preprocessing pipeline created!")

## 4. Handling Class Imbalance

We'll compare several approaches:
1. No handling (baseline)
2. Class weights
3. SMOTE oversampling

In [ ]:
# Calculate class weights
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), class_weights))

print(f"Class weights: {class_weight_dict}")

In [ ]:
# SMOTE implementation (simplified, no imblearn dependency)
def simple_oversample(X, y, target_ratio=0.5, random_state=42):
    """
    Simple random oversampling of minority class.
    """
    np.random.seed(random_state)
    
    minority_mask = y == 1
    majority_mask = y == 0
    
    X_minority = X[minority_mask]
    y_minority = y[minority_mask]
    X_majority = X[majority_mask]
    y_majority = y[majority_mask]
    
    n_majority = len(y_majority)
    n_target = int(n_majority * target_ratio / (1 - target_ratio))
    n_to_sample = n_target - len(y_minority)
    
    if n_to_sample > 0:
        # Random oversampling with replacement
        indices = np.random.choice(len(X_minority), size=n_to_sample, replace=True)
        X_oversampled = np.vstack([X_minority, X_minority.iloc[indices].values if hasattr(X_minority, 'iloc') else X_minority[indices]])
        y_oversampled = np.hstack([y_minority, y_minority.iloc[indices].values if hasattr(y_minority, 'iloc') else y_minority[indices]])
        
        X_resampled = np.vstack([X_majority, X_oversampled])
        y_resampled = np.hstack([y_majority, y_oversampled])
    else:
        X_resampled = np.vstack([X_majority, X_minority])
        y_resampled = np.hstack([y_majority, y_minority])
    
    # Shuffle
    shuffle_idx = np.random.permutation(len(y_resampled))
    return X_resampled[shuffle_idx], y_resampled[shuffle_idx]


# Preprocess data for oversampling
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Apply oversampling
X_train_resampled, y_train_resampled = simple_oversample(
    X_train_processed, y_train.values, target_ratio=0.4
)

print(f"Original training set: {len(y_train)} samples")
print(f"Resampled training set: {len(y_train_resampled)} samples")
print(f"\nOriginal churn rate: {y_train.mean()*100:.1f}%")
print(f"Resampled churn rate: {y_train_resampled.mean()*100:.1f}%")

## 5. Model Training and Comparison

In [ ]:
def train_and_evaluate(model, X_train, X_test, y_train, y_test, name="Model"):
    """
    Train model and return comprehensive metrics.
    """
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred
    
    # Metrics
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    }
    
    return metrics, model, y_pred, y_proba


# Models to compare
results = {}
models = {}
predictions = {}

# 1. Logistic Regression - Baseline
print("Training Logistic Regression (Baseline)...")
lr_baseline = LogisticRegression(random_state=42, max_iter=1000)
results['LR Baseline'], models['LR Baseline'], _, _ = train_and_evaluate(
    lr_baseline, X_train_processed, X_test_processed, y_train, y_test
)

# 2. Logistic Regression - Class Weights
print("Training Logistic Regression (Class Weights)...")
lr_weighted = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
results['LR Weighted'], models['LR Weighted'], _, _ = train_and_evaluate(
    lr_weighted, X_train_processed, X_test_processed, y_train, y_test
)

# 3. Random Forest - Class Weights
print("Training Random Forest (Class Weights)...")
rf_weighted = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
)
results['RF Weighted'], models['RF Weighted'], predictions['RF Weighted'], proba_rf = train_and_evaluate(
    rf_weighted, X_train_processed, X_test_processed, y_train, y_test
)

# 4. Gradient Boosting - Oversampled
print("Training Gradient Boosting (Oversampled)...")
gb_oversampled = GradientBoostingClassifier(n_estimators=100, random_state=42)
results['GB Oversampled'], models['GB Oversampled'], predictions['GB Oversampled'], proba_gb = train_and_evaluate(
    gb_oversampled, X_train_resampled, X_test_processed, y_train_resampled, y_test
)

print("\nTraining complete!")

In [ ]:
# Results comparison
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

print("=== Model Comparison ===")
print(results_df.to_string())

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics comparison
metrics_to_plot = ['Precision', 'Recall', 'F1 Score', 'AUC-ROC']
x = np.arange(len(results_df))
width = 0.2

for i, metric in enumerate(metrics_to_plot):
    axes[0].bar(x + i*width, results_df[metric], width, label=metric)

axes[0].set_ylabel('Score')
axes[0].set_title('Model Metrics Comparison')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(results_df.index, rotation=15)
axes[0].legend()
axes[0].set_ylim(0, 1)

# ROC curves
for name in ['RF Weighted', 'GB Oversampled']:
    model = models[name]
    if name == 'GB Oversampled':
        y_proba = proba_gb
    else:
        y_proba = proba_rf
    
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Threshold Optimization

Default threshold of 0.5 may not be optimal for business needs.

In [ ]:
# Best model for threshold analysis
best_model = models['RF Weighted']
y_proba = proba_rf

# Calculate precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

# Find optimal threshold for F1
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Default threshold: 0.5")
print(f"Optimal F1 threshold: {optimal_threshold:.3f}")
print(f"\nF1 at 0.5: {f1_score(y_test, (y_proba >= 0.5).astype(int)):.4f}")
print(f"F1 at optimal: {f1_scores[optimal_idx]:.4f}")

In [ ]:
# Visualize threshold effects
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision-Recall curve
axes[0].plot(recall, precision)
axes[0].scatter([recall[optimal_idx]], [precision[optimal_idx]], 
               c='red', s=100, label=f'Optimal (t={optimal_threshold:.2f})')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()

# Metrics vs threshold
threshold_range = np.linspace(0.1, 0.9, 50)
precisions = []
recalls = []
f1s = []

for t in threshold_range:
    y_pred_t = (y_proba >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t))
    f1s.append(f1_score(y_test, y_pred_t))

axes[1].plot(threshold_range, precisions, label='Precision')
axes[1].plot(threshold_range, recalls, label='Recall')
axes[1].plot(threshold_range, f1s, label='F1 Score', linewidth=2)
axes[1].axvline(x=optimal_threshold, color='red', linestyle='--', label=f'Optimal ({optimal_threshold:.2f})')
axes[1].axvline(x=0.5, color='gray', linestyle='--', label='Default (0.5)')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Metrics vs Classification Threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Business Impact Analysis

Calculate the cost/benefit of different thresholds.

In [ ]:
# Business assumptions
CUSTOMER_VALUE = 500  # Annual value of customer
RETENTION_COST = 50   # Cost of retention campaign per customer
RETENTION_SUCCESS = 0.4  # Probability retention campaign works

def calculate_business_value(y_true, y_pred_proba, threshold):
    """
    Calculate business value of predictions.
    """
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # Confusion matrix components
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    
    # Calculate values
    # True positives: Saved customers (some retained)
    value_tp = tp * (RETENTION_SUCCESS * CUSTOMER_VALUE - RETENTION_COST)
    
    # False positives: Wasted retention cost
    value_fp = fp * (-RETENTION_COST)
    
    # False negatives: Lost customers
    value_fn = fn * (-CUSTOMER_VALUE)
    
    # True negatives: No action needed (no cost/benefit)
    value_tn = 0
    
    total_value = value_tp + value_fp + value_fn + value_tn
    
    return {
        'threshold': threshold,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'value_tp': value_tp,
        'value_fp': value_fp,
        'value_fn': value_fn,
        'total_value': total_value
    }


# Analyze different thresholds
thresholds_to_analyze = [0.2, 0.3, 0.4, 0.5, 0.6]
business_results = []

for t in thresholds_to_analyze:
    result = calculate_business_value(y_test.values, y_proba, t)
    business_results.append(result)

business_df = pd.DataFrame(business_results)
print("=== Business Impact by Threshold ===")
print(business_df[['threshold', 'tp', 'fp', 'fn', 'total_value']].to_string(index=False))
print(f"\nBest threshold for business value: {business_df.loc[business_df['total_value'].idxmax(), 'threshold']}")

In [ ]:
# Full threshold sweep for business value
threshold_range = np.linspace(0.1, 0.9, 50)
business_values = []

for t in threshold_range:
    result = calculate_business_value(y_test.values, y_proba, t)
    business_values.append(result['total_value'])

# Find optimal business threshold
best_business_idx = np.argmax(business_values)
best_business_threshold = threshold_range[best_business_idx]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(threshold_range, business_values, linewidth=2)
plt.axvline(x=best_business_threshold, color='green', linestyle='--', 
            label=f'Best Business (t={best_business_threshold:.2f})')
plt.axvline(x=optimal_threshold, color='red', linestyle='--', 
            label=f'Best F1 (t={optimal_threshold:.2f})')
plt.axvline(x=0.5, color='gray', linestyle='--', label='Default (t=0.5)')
plt.xlabel('Threshold')
plt.ylabel('Business Value ($)')
plt.title('Business Value vs Classification Threshold')
plt.legend()
plt.grid(True)
plt.show()

print(f"\nOptimal Business Threshold: {best_business_threshold:.3f}")
print(f"Maximum Business Value: ${business_values[best_business_idx]:,.0f}")

## 8. Feature Importance and Insights

In [ ]:
# Get feature names
cat_encoder = preprocessor.named_transformers_['categorical'].named_steps['encoder']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_feature_names = numerical_features + cat_feature_names + binary_features

# Feature importance from Random Forest
importances = models['RF Weighted'].feature_importances_

importance_df = pd.DataFrame({
    'feature': all_feature_names,
    'importance': importances
}).sort_values('importance', ascending=True)

# Plot
plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if 'contract' in f or 'tenure' in f else '#3498db' 
          for f in importance_df['feature']]
plt.barh(importance_df['feature'], importance_df['importance'], color=colors)
plt.xlabel('Importance')
plt.title('Feature Importance for Churn Prediction')
plt.tight_layout()
plt.show()

print("\n=== Top 5 Churn Predictors ===")
for _, row in importance_df.tail(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

In [ ]:
# Final confusion matrix with optimal threshold
y_pred_optimal = (y_proba >= best_business_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_optimal)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Churned', 'Churned'],
            yticklabels=['Not Churned', 'Churned'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix (Threshold = {best_business_threshold:.2f})')
plt.tight_layout()
plt.show()

print("\n=== Classification Report (Optimal Threshold) ===")
print(classification_report(y_test, y_pred_optimal, target_names=['Not Churned', 'Churned']))

## 9. Save Model and Generate Insights Report

In [ ]:
import joblib

# Save model and preprocessor
model_path = Path('./models')
model_path.mkdir(exist_ok=True)

joblib.dump({
    'preprocessor': preprocessor,
    'model': models['RF Weighted'],
    'optimal_threshold': best_business_threshold,
    'feature_names': all_feature_names
}, model_path / 'churn_model.joblib')

print(f"Model saved to {model_path / 'churn_model.joblib'}")

In [ ]:
# Business insights summary
print("=" * 60)
print("CUSTOMER CHURN ANALYSIS - EXECUTIVE SUMMARY")
print("=" * 60)

print("\n📊 KEY FINDINGS:")
print(f"   • Total customers analyzed: {len(df):,}")
print(f"   • Current churn rate: {df['churned'].mean()*100:.1f}%")
print(f"   • At-risk customers identified: {(y_proba >= best_business_threshold).sum():,}")

print("\n🎯 TOP CHURN RISK FACTORS:")
top_factors = importance_df.tail(3)
for _, row in top_factors.iterrows():
    print(f"   • {row['feature'].replace('_', ' ').title()}")

print("\n💰 BUSINESS RECOMMENDATIONS:")
print("   1. Focus retention on month-to-month contract customers")
print("   2. Target customers with high support ticket counts")
print("   3. Offer incentives for long-term contracts")
print("   4. Promote security add-on services (reduces churn)")

print("\n📈 MODEL PERFORMANCE:")
print(f"   • AUC-ROC: {results['RF Weighted']['AUC-ROC']:.3f}")
print(f"   • Optimal threshold: {best_business_threshold:.2f}")
print(f"   • Expected annual savings: ${business_values[best_business_idx]:,.0f}")

print("\n" + "=" * 60)

## Summary

### Skills Demonstrated

- **Imbalanced data handling**: Class weights, oversampling
- **Threshold optimization**: Beyond default 0.5
- **Business value analysis**: Cost-benefit framework
- **Model interpretation**: Feature importance, actionable insights
- **Production considerations**: Model serialization, business reporting